TravelGenie AI Pro

            User
              │
              ▼
       Chat Interface
              │
              ▼
        Memory Manager
              │
              ▼
     Intent Classification
              │
              ▼
     Multi-Agent Controller
              │
 ┌────────────┼────────────┐
 ▼            ▼            ▼
Planner   Research    Weather
 │            │            │
 ▼            ▼            ▼
Hotel     Budget      Food
 │            │            │
 └────────────┼────────────┘
              ▼
      Itinerary Agent
              ▼
      Reviewer Agent
              ▼
        Final Response

💬 Chat interface (gr.ChatInterface)
🧠 Conversation memory
🌦 Live weather API
🏨 Hotel recommendations
🍽 Restaurant suggestions
🚆 Bus/Train recommendations
🗺 Google Maps integration (later)
📄 PDF itinerary generation
💾 Save travel history

Intelligent Multi-Agent Travel Assistant

New Features

💬 ChatGPT-style chat interface
🧠 Conversation memory
🎯 Intent classification
🤖 AI Router Agent
🌦 Weather Agent (API later)
🏨 Hotel Agent
🍽 Restaurant Agent
💰 Budget Agent
📅 Itinerary Agent
✅ Reviewer Agent

Architecture

                   User
                     │
                     ▼
             Chat Interface
                     │
                     ▼
             Memory Manager
                     │
                     ▼
          Intent Classifier
                     │
                     ▼
              Router Agent
                     │
 ┌─────────────┬─────────────┬─────────────┐
 ▼             ▼             ▼             ▼
Planner     Research      Budget      Itinerary
                     │
                     ▼
               Hotel Agent
                     │
                     ▼
             Restaurant Agent
                     │
                     ▼
              Reviewer Agent
                     │
                     ▼
               Final Response

In [1]:
!pip install -q openai gradio requests pandas

In [2]:
import gradio as gr
import requests
import pandas as pd

from openai import OpenAI
from google.colab import userdata

In [4]:
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

Memory Manager

Instead of a simple dictionary, we'll use a structured memory.

In [5]:
memory = {
    "conversation": [],
    "destination": "",
    "days": "",
    "budget": "",
    "travel_type": "",
    "history": []
}

Memory Functions

In [6]:
def add_memory(role, message):
    memory["conversation"].append(
        {
            "role": role,
            "message": message
        }
    )

def get_history():

    text = ""

    for chat in memory["conversation"]:

        text += f"{chat['role']}: {chat['message']}\n"

    return text

Test Memory

In [7]:
add_memory("User","Plan trip to Ooty")

add_memory("Assistant","Sure")

print(memory)

{'conversation': [{'role': 'User', 'message': 'Plan trip to Ooty'}, {'role': 'Assistant', 'message': 'Sure'}], 'destination': '', 'days': '', 'budget': '', 'travel_type': '', 'history': []}


Intent Classifier Agent

Instead of always calling every agent, it first understands what the user wants.

In [8]:
def intent_classifier(user_query):

    prompt = f"""
You are an Intent Classification Agent.

Classify the user's request into ONLY ONE of the following intents.

Available Intents:

PLAN_TRIP
WEATHER
HOTEL
RESTAURANT
BUDGET
ITINERARY
TRANSPORT
PACKING
GENERAL

Return ONLY the intent name.

User Request:

{user_query}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "You are an Intent Classification Agent."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
        max_tokens=20
    )

    intent = response.choices[0].message.content.strip().upper()

    memory["intent"] = intent

    return intent

In [9]:
query = "Plan a 5 day family trip to Ooty"

intent = intent_classifier(query)

print(intent)

PLAN_TRIP


In [10]:
query = "Show me hotels in Ooty"

intent = intent_classifier(query)

print(intent)

HOTEL


In [11]:
query = "What's the weather in Kodaikanal?"

intent = intent_classifier(query)

print(intent)

WEATHER


In [12]:
query = "How much budget is needed for Goa?"

intent = intent_classifier(query)

print(intent)

BUDGET


In [13]:
query = "Suggest restaurants in Mysore"

intent = intent_classifier(query)

print(intent)

RESTAURANT


How It Works
User

↓

Plan 5 days in Ooty

↓

Intent Classifier

↓

PLAN_TRIP

↓

Planner Agent

User

↓

Hotels in Ooty

↓

Intent Classifier

↓

HOTEL

↓

Hotel Agent

User

↓

Weather in Chennai

↓

Intent Classifier

↓

WEATHER

↓

Weather Agent

Memory

In [14]:
print(memory)

{'conversation': [{'role': 'User', 'message': 'Plan trip to Ooty'}, {'role': 'Assistant', 'message': 'Sure'}], 'destination': '', 'days': '', 'budget': '', 'travel_type': '', 'history': [], 'intent': 'RESTAURANT'}


This reduces:

🚀 Response time
💰 LLM token usage
⚡ Cost
🧠 Complexity

Router Agent.

The Router Agent uses the intent from the Intent Classifier to decide which specialized agent to call:

The Router Agent decides which specialized agent to call.

In [19]:
def router_agent(user_query):

    # Step 1: Identify user intent
    intent = intent_classifier(user_query)

    print(f"Detected Intent: {intent}")

    # Step 2: Route to the correct agent

    if intent == "PLAN_TRIP":

        task = user_query

        plan = planner_agent(task)

        research = research_agent(task, plan)

        budget = budget_agent(task, research)

        itinerary = itinerary_agent(
            task,
            research,
            budget
        )

        review = reviewer_agent(
            plan,
            research,
            budget,
            itinerary
        )

        return f"""
# 🌍 TravelGenie AI

## 📋 Execution Plan

{plan}

----------------------------------------

## 🔍 Destination Research

{research}

----------------------------------------

## 💰 Budget Estimate

{budget}

----------------------------------------

## 📅 Itinerary

{itinerary}

----------------------------------------

## ✅ Review

{review}
"""

    elif intent == "WEATHER":

        return "🌦 Weather Agent will be implemented in the next version."

    elif intent == "HOTEL":

        return "🏨 Hotel Agent will be implemented in the next version."

    elif intent == "RESTAURANT":

        return "🍽 Restaurant Agent will be implemented in the next version."

    elif intent == "TRANSPORT":

        return "🚌 Transport Agent will be implemented in the next version."

    elif intent == "PACKING":

        return "🎒 Packing Agent will be implemented in the next version."

    elif intent == "BUDGET":

        return "💰 Budget Assistant will be implemented in the next version."

    elif intent == "ITINERARY":

        return "📅 Itinerary Assistant will be implemented in the next version."

    else:

        return "🤖 I can help you plan trips, budgets, hotels, restaurants, weather, and itineraries."

Planner Agent V2

In [18]:
def planner_agent(user_query):

    history = get_history()

    prompt = f"""
You are an expert Travel Planning AI.

Your job is to analyze the user's request and create a structured travel plan.

Conversation History:
{history}

Current Request:
{user_query}

Create the travel plan in this format:

Destination:
Number of Days:
Travel Type:
Estimated Budget (if mentioned):
Main Objectives:
Suggested Planning Steps:

Do not create the itinerary yet.
Do not estimate hotels or restaurants.
Only create a high-level execution plan.
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "You are an expert travel planning assistant."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.3
    )

    plan = response.choices[0].message.content.strip()

    memory["plan"] = plan

    add_memory("User", user_query)
    add_memory("Planner", plan)

    return plan

In [20]:
query = """
Plan a 5-day family trip to Ooty with a budget of ₹30,000.
"""

plan = planner_agent(query)

print(plan)

Destination: Ooty
Number of Days: 5
Travel Type: Family Trip
Estimated Budget: ₹30,000
Main Objectives: To plan a 5-day family trip to Ooty within the given budget, ensuring a memorable and enjoyable experience for all family members.
Suggested Planning Steps:
1. Research and identify must-visit attractions and activities in Ooty suitable for families.
2. Determine the best mode of transportation to and within Ooty, considering factors like cost, convenience, and travel time.
3. Explore accommodation options that fit within the budget and cater to family needs.
4. Plan the daily schedule, allocating time for sightseeing, relaxation, and other activities.
5. Identify any additional expenses, such as food, entry fees, and miscellaneous costs, to ensure the overall budget is not exceeded.
6. Consider the best time to visit Ooty, taking into account weather conditions and tourist season.
7. Finalize the travel dates and make necessary bookings and reservations in advance to avoid last-minu

In [21]:
print(memory)

{'conversation': [{'role': 'User', 'message': 'Plan trip to Ooty'}, {'role': 'Assistant', 'message': 'Sure'}, {'role': 'User', 'message': '\nPlan a 5-day family trip to Ooty with a budget of ₹30,000.\n'}, {'role': 'Planner', 'message': 'Destination: Ooty\nNumber of Days: 5\nTravel Type: Family Trip\nEstimated Budget: ₹30,000\nMain Objectives: To plan a 5-day family trip to Ooty within the given budget, ensuring a memorable and enjoyable experience for all family members.\nSuggested Planning Steps:\n1. Research and identify must-visit attractions and activities in Ooty suitable for families.\n2. Determine the best mode of transportation to and within Ooty, considering factors like cost, convenience, and travel time.\n3. Explore accommodation options that fit within the budget and cater to family needs.\n4. Plan the daily schedule, allocating time for sightseeing, relaxation, and other activities.\n5. Identify any additional expenses, such as food, entry fees, and miscellaneous costs, to

Research Agent V2

In [22]:
def research_agent(user_query, plan):

    history = get_history()

    prompt = f"""
You are an expert Travel Research AI.

Conversation History:
{history}

Travel Plan:
{plan}

Current User Request:
{user_query}

Provide detailed destination research using the following format:

Destination Overview

Best Time to Visit

Top Tourist Attractions
- Attraction 1
- Attraction 2
- Attraction 3
- Attraction 4
- Attraction 5

Popular Local Foods

Shopping Areas

Local Transportation

Travel Tips

Keep the response practical and easy to read.
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "You are an expert travel research assistant."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.4
    )

    research = response.choices[0].message.content.strip()

    memory["research"] = research

    add_memory("Research", research)

    return research

Test Research Agent

In [23]:
query = "Plan a 5-day family trip to Ooty with a budget of ₹30,000."

plan = planner_agent(query)

research = research_agent(query, plan)

print(research)

**Destination Overview**
Ooty, also known as Udhagamandalam, is a popular hill station in the state of Tamil Nadu, India. It is situated in the Nilgiri Hills, at an altitude of 2,240 meters above sea level. Ooty is known for its scenic beauty, pleasant climate, and rich cultural heritage. The town is surrounded by lush green forests, tea plantations, and eucalyptus trees, making it a perfect destination for nature lovers and those seeking a relaxing getaway.

**Best Time to Visit**
The best time to visit Ooty is from October to February, when the weather is cool and pleasant, with average temperatures ranging from 10°C to 20°C. This period is ideal for sightseeing and outdoor activities. However, if you're looking for a budget-friendly option, consider visiting during the off-season (March to May or September to November), when the weather is still pleasant and the crowds are smaller.

**Top Tourist Attractions**
- **Ooty Lake**: A picturesque lake with boating facilities, surrounded b

In [24]:
print(memory.keys())

dict_keys(['conversation', 'destination', 'days', 'budget', 'travel_type', 'history', 'intent', 'plan', 'research'])


This version is more structured than V1 and uses:

✅ User request
✅ Planner output
✅ Research output
✅ Conversation history
✅ Memory

Budget Agent V2

In [25]:
def budget_agent(user_query, plan, research):

    history = get_history()

    prompt = f"""
You are an expert Travel Budget Planner.

Conversation History:
{history}

Travel Plan:
{plan}

Destination Research:
{research}

Current User Request:
{user_query}

Estimate the travel budget using the following format.

Transportation
- Estimated Cost

Accommodation
- Budget Hotel
- Standard Hotel
- Luxury Hotel

Food
- Daily Food Cost
- Total Food Cost

Sightseeing
- Entry Fees

Shopping
- Recommended Shopping Budget

Miscellaneous
- Emergency Expenses

Grand Total

Finally provide:

Budget Saving Tips
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "You are an expert travel budget planner."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.3
    )

    budget = response.choices[0].message.content.strip()

    memory["budget"] = budget

    add_memory("Budget", budget)

    return budget

Test Budget Agent

In [26]:
query = "Plan a 5-day family trip to Ooty with a budget of ₹30,000."

plan = planner_agent(query)

research = research_agent(query, plan)

budget = budget_agent(
    query,
    plan,
    research
)

print(budget)

Based on the research and planning, here is an estimated breakdown of the travel budget for a 5-day family trip to Ooty with a budget of ₹30,000:

**Transportation**
- Estimated Cost: ₹4,000 (including to and from Ooty, and local travel)

**Accommodation**
- Budget Hotel: ₹1,500 per night (average)
- Standard Hotel: ₹2,500 per night (average)
- Luxury Hotel: ₹4,000 per night (average)
- Recommended: Budget Hotel (₹1,500 per night) for a total of ₹7,500 for 5 nights

**Food**
- Daily Food Cost: ₹1,600 per day (average)
- Total Food Cost: ₹8,000 for 5 days

**Sightseeing**
- Entry Fees:
  - Ooty Lake: ₹50 per person
  - Botanical Gardens: ₹30 per person
  - Doddabetta Peak: ₹20 per person
  - Wenlock Downs: free entry
  - Tea Factory and Museum: ₹50 per person
- Total Entry Fees: ₹1,500 for a family of 4

**Shopping**
- Recommended Shopping Budget: ₹2,000 for souvenirs and local handicrafts

**Miscellaneous**
- Emergency Expenses: ₹1,000 for unexpected costs

**Grand Total**
- Transporta

In [27]:
print(memory.keys())

dict_keys(['conversation', 'destination', 'days', 'budget', 'travel_type', 'history', 'intent', 'plan', 'research'])


Hotel Agent V2

In [28]:
def hotel_agent(user_query, plan, budget):

    history = get_history()

    prompt = f"""
You are an expert Hotel Recommendation AI.

Conversation History:
{history}

Travel Plan:
{plan}

Budget Information:
{budget}

Current User Request:
{user_query}

Recommend hotels in the following format.

Destination

🏨 Budget Hotels (3)
- Hotel Name
- Approx Price per Night
- Key Features

🏨 Standard Hotels (3)
- Hotel Name
- Approx Price per Night
- Key Features

🏨 Luxury Hotels (3)
- Hotel Name
- Approx Price per Night
- Key Features

Finally provide:

Best Recommendation
Reason for Recommendation

Important:
Use realistic hotel names whenever possible.
Keep prices approximate.
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "You are an expert hotel recommendation assistant."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.4
    )

    hotels = response.choices[0].message.content.strip()

    memory["hotels"] = hotels

    add_memory("Hotel", hotels)

    return hotels

Test Hotel Agent

In [32]:
query = "Plan a 5-day family trip to Ooty with a budget of ₹30,000."

plan = planner_agent(query)

research = research_agent(query, plan)

budget = budget_agent(
    query,
    plan,
    research
)

hotels = hotel_agent(
    query,
    plan,
    budget
)

print(hotels)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kwpv7mvdepcrwjrs1r3cqdgm` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 96908, Requested 7414. Please try again in 1h2m14.208s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}